# Stage 5: Feature Engineering
Feature engineering creates useful input variables from existing data. We must only create features that are supported by the dataset and available at prediction time.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
import pandas as pd
df = pd.read_csv(DATA_DIR / "used_phones_cleaned.csv")
df.head()

,device_brand,os,screen_size,4g,5g,rear_camera_mp,front_camera_mp,internal_memory,ram,battery,weight,release_year,days_used,normalized_used_price,normalized_new_price
0,Honor,Android,14.50,yes,no,13.0,5.0,64.0,3.0,3020.0,146.0,2020,127,4.307572,4.715100
1,Honor,Android,17.30,yes,yes,13.0,16.0,128.0,8.0,4300.0,213.0,2020,325,5.162097,5.519018
2,Honor,Android,16.69,yes,yes,13.0,8.0,128.0,8.0,4200.0,213.0,2020,162,5.111084,5.884631
3,Honor,Android,25.50,yes,yes,13.0,8.0,64.0,6.0,7250.0,480.0,2020,345,5.135387,5.630961
4,Honor,Android,15.32,yes,no,13.0,8.0,64.0,3.0,5000.0,185.0,2020,293,4.389995,4.947837


## Example: derive phone age from release year
Use a consistent reference year. The notebook uses the current year dynamically. Check the dataset's release-year values first. This feature is optional and should be retained only if it makes sense for the dataset.

In [2]:
from datetime import date
if "release_year" in df.columns:
    current_year = date.today().year
    df["phone_age_years"] = (current_year - pd.to_numeric(df["release_year"], errors="coerce")).clip(lower=0)
    display(df[["release_year", "phone_age_years"]].head())
    # Avoid redundant inputs: retain phone_age_years and remove release_year.
    df = df.drop(columns=["release_year"])
else:
    print("release_year not present; skip phone_age_years.")

,release_year,phone_age_years
0,2020,6
1,2020,6
2,2020,6
3,2020,6
4,2020,6


## Avoid data leakage
Do not include the used price in X. Do not derive features using information that would only be known after the resale transaction. Do not create made-up battery health or physical condition values when the dataset does not contain them.

In [3]:
TARGET = "normalized_used_price"
X = df.drop(columns=[TARGET])
y = df[TARGET]
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", X.columns.tolist())

Feature shape: (3454, 14)
Target shape: (3454,)
Features: ['device_brand', 'os', 'screen_size', '4g', '5g', 'rear_camera_mp', 'front_camera_mp', 'internal_memory', 'ram', 'battery', 'weight', 'days_used', 'normalized_new_price', 'phone_age_years']


In [4]:
# Save the engineered working dataset if you decide to keep the derived feature.
df.to_csv(DATA_DIR / "used_phones_features.csv", index=False)
print("Saved engineered dataset.")

Saved engineered dataset.
